# PMSA v2 — Phase 1 extraction smoke test (Kaggle T4)

Goal: validate the `extract → cache → baseline` loop on a **small** GenImage subset
before scaling. Start with 2 generators × 2k images/class. If the UnivFD baseline
AUC looks sane (in-distribution should be high; leave-one-generator-out is the real
test), the pipeline works and you can scale `PER_CLASS_LIMIT`.

**Settings:** turn ON GPU and Internet in the notebook settings panel. Use a **T4**
(single or T4 ×2) — **not the P100**: Kaggle's PyTorch dropped support for the
P100's sm_60 compute capability, so it warns and may fail. The T4 is sm_75 and runs
clean. The pipeline uses a single GPU, so T4 ×2 just leaves one card idle (harmless).
Add the GenImage dataset to the notebook's inputs.

In [ ]:
# 1) Get the code. Public repo — clone and install (core only; torch/transformers
#    are already on the Kaggle image).
!git clone -q https://github.com/Y-ash-Y/AI-Detection-.git pmsa_v2 || (cd pmsa_v2 && git pull -q)
%cd pmsa_v2
!pip install -q -e . transformers --no-deps 2>/dev/null; pip install -q pyyaml joblib scikit-learn
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2) Point to the GenImage input and confirm the layout.
#    Expected: <ROOT>/<generator>/<split>/{ai,nature}/*.
#    Adjust GENIMAGE_ROOT to match your added dataset (check /kaggle/input).
import os, glob
for p in sorted(glob.glob('/kaggle/input/*')):
    print(p)

GENIMAGE_ROOT = '/kaggle/input/genimage'   # <-- EDIT to your dataset path
GENERATORS    = ['stable_diffusion_v_1_4', 'sdxl']  # <-- 2 to start; EDIT names to match folders
PER_CLASS_LIMIT = 2000
print('\ngenerators present:', sorted(os.listdir(GENIMAGE_ROOT))[:20])

In [ ]:
# 3) Extraction pass: writes one npz per backbone into feature_cache/ (tag=train).
#    First run downloads CLIP-L/14 (~1.7GB) + DINOv2-base; needs Internet ON.
!python scripts/01_extract_features.py \
    --genimage-root {GENIMAGE_ROOT} \
    --generators {' '.join(GENERATORS)} \
    --per-class-limit {PER_CLASS_LIMIT} \
    --split train --tag train --device cuda

In [ ]:
# 4) Sanity-check the caches: shapes, label balance, alignment across backbones.
from pmsa.features import FeatureSet
import numpy as np, glob
sets = {}
for f in sorted(glob.glob('feature_cache/*_train.npz')):
    fs = FeatureSet.load(f); sets[fs.backbone] = fs
    print(f'{fs.backbone:10s} feats={fs.features.shape} '
          f'real={int((fs.labels==0).sum())} fake={int((fs.labels==1).sum())} '
          f'gens={sorted(set(fs.source[fs.labels==1]))}')
names = list(sets)
if len(names) > 1:
    base = sets[names[0]]
    aligned = all(np.array_equal(sets[n].paths, base.paths) for n in names[1:])
    print('caches aligned across backbones:', aligned)

In [ ]:
# 5) The floor: UnivFD linear probe on frozen CLIP, leave-one-generator-out.
#    Each held-out generator is the zero-shot test. This is the number v2 must beat.
!python scripts/02_train_baseline.py --backbone clip_l14 --tag train
import json
r = json.load(open('outputs/baseline_clip_l14_logo.json'))
for g, rep in r.items():
    print(f"held-out {g:24s} AUC={rep['auc']['point']:.4f} "
          f"[{rep['auc']['lo']:.3f},{rep['auc']['hi']:.3f}]  "
          f"TPR@1%={rep['tpr_at_0.01']['point']:.4f}")

## What to look for

- **Caches aligned = True** and equal row counts per backbone — if not, the
  dataloader order broke; fusion can't fuse misaligned streams.
- **LOGO AUC** well above 0.5 means the frozen-CLIP baseline transfers to a held-out
  generator. With only 2 generators this is a weak test; the point here is that the
  *loop runs end to end*, not the final number.

## Next once green

1. Scale: `PER_CLASS_LIMIT = 10000`, add more generators, re-extract.
2. Extract a `--tag test` set that also includes a **second real domain**
   (`--real-dir ffhq:/path`) and in-the-wild fakes (`--fake-dir nano_banana:/path`)
   so the shift matrix (scripts 04–05) has cells C/D to fill.
3. Save `feature_cache/*.npz` as a Kaggle **dataset output** so you can train on the
   M4 without re-running extraction.
4. Phase 3: `scripts/03_train_fusion.py` (runs fine on the laptop from the caches).